## Hybrid model pipeline: Rule based logic + statistical classification

In [ ]:
import os
import re
import unicodedata
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.svm import LinearSVC

In [ ]:
KAGGLE_DIR = '/kaggle/input/uab-asho-ai-codification'
LOCAL_DIR  = 'data/raw'

if os.path.isdir(KAGGLE_DIR):
    base, train_file, test_file = KAGGLE_DIR, 'codification_data.csv', 'leaderboard_data.csv'
else:
    base, train_file, test_file = LOCAL_DIR, 'training_codification_data.csv', 'test_leaderboard_data.csv'

train_raw = pd.read_csv(f'{base}/{train_file}')
icd_raw   = pd.read_csv(f'{base}/icd_d_p_pairs.csv')
test_raw  = pd.read_csv(f'{base}/{test_file}')

# CodiEsp (zenodo.org/records/3837305): Spanish clinical (mention -> ICD-10) pairs.
# Adds ~8k real clinical literals to training; worth +1.6pp on local CV.
# The notebook works without it; to enable on Kaggle, attach the dataset.
codiesp = None
for cand in [
    'data/codiesp/final_dataset_v4_to_publish',
    '/kaggle/input/codiesp/final_dataset_v4_to_publish',
    '/kaggle/input/codiesp',
]:
    if os.path.isfile(f'{cand}/train/trainX.tsv'):
        cols = ['articleID', 'label', 'Code', 'Literal', 'pos']
        codiesp = pd.concat([
            pd.read_csv(f'{cand}/train/trainX.tsv', sep='\t', header=None, names=cols),
            pd.read_csv(f'{cand}/dev/devX.tsv',     sep='\t', header=None, names=cols),
            pd.read_csv(f'{cand}/test/testX.tsv',   sep='\t', header=None, names=cols),
        ], ignore_index=True)
        codiesp['Code'] = codiesp['Code'].astype(str).str.upper().str.replace('.', '', regex=False)
        codiesp = codiesp[['Literal', 'Code']].drop_duplicates().reset_index(drop=True)
        print(f'CodiEsp loaded from {cand}: {len(codiesp):,} unique pairs')
        break
if codiesp is None:
    print('CodiEsp not found - running without it (~1.6pp lower CV score).')

print('train:', train_raw.shape, '| icd:', icd_raw.shape, '| test:', test_raw.shape)
test_raw.head()

In [ ]:
ABBREVIATIONS = {
    'HTA':    'hipertension arterial',
    'IAM':    'infarto agudo miocardio',
    'ACV':    'accidente cerebrovascular',
    'FA':     'fibrilacion auricular',
    'IC':     'insuficiencia cardiaca',
    'ICC':    'insuficiencia cardiaca congestiva',
    'TEP':    'tromboembolismo pulmonar',
    'TVP':    'trombosis venosa profunda',
    'SCA':    'sindrome coronario agudo',
    'IRC':    'insuficiencia renal cronica',
    'IRA':    'insuficiencia renal aguda',
    'RAO':    'retencion aguda orina',
    'EPOC':   'enfermedad pulmonar obstructiva cronica',
    'SAHS':   'sindrome apnea hipopnea sueno',
    'SAOS':   'sindrome apnea obstructiva sueno',
    'CA':     'cancer',
    'NEO':    'neoplasia',
    'LMA':    'leucemia mieloide aguda',
    'LLC':    'leucemia linfocitica cronica',
    'LNH':    'linfoma no hodgkin',
    'GEA':    'gastroenteritis aguda',
    'ERGE':   'enfermedad reflujo gastroesofagico',
    'HDA':    'hemorragia digestiva alta',
    'HDB':    'hemorragia digestiva baja',
    'VHB':    'virus hepatitis B',
    'VHC':    'virus hepatitis C',
    'VHA':    'virus hepatitis A',
    'TCE':    'traumatismo craneoencefalico',
    'TIA':    'accidente isquemico transitorio',
    'AIT':    'accidente isquemico transitorio',
    'DM':     'diabetes mellitus',
    'DM2':    'diabetes mellitus tipo 2',
    'DM1':    'diabetes mellitus tipo 1',
    'DMG':    'diabetes mellitus gestacional',
    'HLD':    'hiperlipidemia',
    'DLP':    'dislipemia',
    'IQ':     'intervencion quirurgica',
    'RN':     'recien nacido',
    'RNPT':   'recien nacido prematuro',
    'RPM':    'rotura prematura membranas',
    'ITU':    'infeccion tracto urinario',
    'PCR':    'parada cardiorrespiratoria',
    'VIH':    'virus inmunodeficiencia humana',
    'CX':     'cirugia',
    'RX':     'radiografia',
    'TAC':    'tomografia axial computarizada',
    'RM':     'resonancia magnetica',
    'SF':     'sindrome febril',
    'SD':     'sindrome',
    'ANTI-D': 'inmunoglobulina anti D',
}

CATALAN_TO_SPANISH = {
    'migranya':      'migranya',
    'cos estrany':   'cuerpo extranos',
    'cos':           'cuerpo',
    'estrany':       'extranos',
    'part':          'parto',
    'malaltia':      'enfermedad',
    'infeccio':      'infeccion',
    'embaras':       'embarazo',
    'diabetis':      'diabetes',
    'hipertensio':   'hipertension',
    'pneumonia':     'neumonia',
    'limfoma':       'linfoma',
    'cataracta':     'catarata',
    'artrosi':       'artrosis',
    'osteoporosi':   'osteoporosis',
    'calcul':        'calculo',
    'dret':          'derecho',
    'dreta':         'derecha',
    'esquerra':      'izquierda',
    'agut':          'agudo',
    'aguda':         'aguda',
    'cronic':        'cronico',
    'cronica':       'cronica',
    'amniodrenatge': 'amniodrenaje',
    'laceracio':     'laceracion',
    'laceracions':   'laceraciones',
    'postpart':      'postparto',
    'cesaria':       'cesarea',
}

ICD10_FULL   = re.compile(r'^[A-Z]\d{2}\.?\d*[A-Z0-9]*$')
ICD10_INLINE = re.compile(r'\b([A-Z]\d{2}\.?\d*[A-Z0-9]*)\b')


def remove_accents(text):
    return unicodedata.normalize('NFC',
        ''.join(c for c in unicodedata.normalize('NFD', text)
                if unicodedata.category(c) != 'Mn'))


def apply_catalan_dict(text):
    for cat, esp in sorted(CATALAN_TO_SPANISH.items(), key=lambda x: -len(x[0])):
        text = re.sub(r'\b' + re.escape(cat) + r'\b', esp, text, flags=re.IGNORECASE)
    return text


def apply_abbreviations(text):
    return ' '.join(ABBREVIATIONS.get(tok.upper(), tok) for tok in text.split())


def preprocess(text):
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.strip().lower()
    text = remove_accents(text)
    text = apply_catalan_dict(text)
    text = apply_abbreviations(text)
    return re.sub(r'\s+', ' ', text).strip()


def extract_icd_from_literal(literal):
    if not isinstance(literal, str):
        return None
    text = literal.strip().upper()
    if ICD10_FULL.match(text):
        return text.replace('.', '')
    m = ICD10_INLINE.search(text)
    if m:
        return m.group(1).replace('.', '')
    return None

In [ ]:
train_raw['processed']   = train_raw['Literal'].fillna('').apply(preprocess)
test_raw['processed']    = test_raw['Literal'].fillna('').apply(preprocess)
test_raw['icd_shortcut'] = test_raw['Literal'].apply(extract_icd_from_literal)
icd_raw['processed']     = icd_raw['Description'].fillna('').apply(preprocess)
icd_raw = icd_raw[icd_raw['processed'] != ''].dropna(subset=['Code']).reset_index(drop=True)

if codiesp is not None:
    codiesp['processed'] = codiesp['Literal'].fillna('').apply(preprocess)
    codiesp = codiesp[codiesp['processed'] != ''].dropna(subset=['Code']).reset_index(drop=True)

print('icd_shortcut hits:', test_raw['icd_shortcut'].notna().sum(), '/', len(test_raw))
test_raw[['Literal', 'processed', 'icd_shortcut']].head(8)

In [ ]:
kb_vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=10000)
kb_matrix = kb_vectorizer.fit_transform(icd_raw['processed'])

test_vecs = kb_vectorizer.transform(test_raw['processed'])
best_idx = cosine_similarity(test_vecs, kb_matrix).argmax(axis=1)
test_raw['sim_code'] = icd_raw['Code'].values[best_idx]

test_raw[['Literal', 'processed', 'sim_code']].head(8)

In [ ]:
# Stack train_raw + icd_raw + CodiEsp (if available).
#   train_raw : 13.7k Kaggle (Literal, Code) — matches test-set shape exactly
#   icd_raw   : 180k official (Description, Code) — broad vocabulary coverage
#   codiesp   : 8.4k clinical (Mention, Code) — real literals, +1.6pp on local CV
Xs = [train_raw['processed'].fillna('').values]
ys = [train_raw['Code'].astype(str).str[0].values]
Xs.append(icd_raw['processed'].values)
ys.append(icd_raw['Code'].astype(str).str[0].values)
if codiesp is not None:
    Xs.append(codiesp['processed'].values)
    ys.append(codiesp['Code'].astype(str).str[0].values)

X_train = np.concatenate(Xs)
y_train = np.concatenate(ys)
mask = (y_train != '') & (X_train != '')
X_train, y_train = X_train[mask], y_train[mask]

# Word + char_wb features. Char n-grams catch medical morphology (-itis, -ectomia, ...).
# Hyperparameters tuned on a held-out 20% of train_raw.
vectorizer = FeatureUnion([
    ('word', TfidfVectorizer(ngram_range=(1, 2), max_features=20000, sublinear_tf=True)),
    ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5),
                              max_features=40000, sublinear_tf=True, min_df=2)),
])
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(test_raw['processed'].fillna(''))

# No class_weight: icd_raw is procedure-heavy (0/S/T) while the leaderboard mirrors
# train_raw (Z/O-heavy). 'balanced' would re-weight in the wrong direction.
model = LinearSVC(C=0.7)
model.fit(X_train_vec, y_train)
test_raw['lr_cat'] = model.predict(X_test_vec)

print(f'Training rows: {len(X_train):,}  (sources: {len(Xs)})')
print(f'Classes seen:  {len(np.unique(y_train))}')

In [ ]:
def pick(row):
    s = row['icd_shortcut']
    if isinstance(s, str) and s:
        return s[0]
    lr = row['lr_cat']
    if isinstance(lr, str) and lr:
        return lr
    sim = row['sim_code']
    if isinstance(sim, str) and sim:
        return sim[0]
    return 'null'

test_raw['y_category'] = test_raw.apply(pick, axis=1)

submission = test_raw[['id', 'Literal', 'y_category']].copy()

assert list(submission.columns) == ['id', 'Literal', 'y_category'], 'Wrong column order'
assert submission['id'].notna().all(), 'Missing id'
assert submission['Literal'].notna().all(), 'Missing Literal'
assert submission['y_category'].notna().all() and (submission['y_category'] != '').all(), \
    'Empty y_category'
assert len(submission) == len(test_raw), 'Row count changed'

submission.to_csv('submission.csv', index=False)

print('Wrote', len(submission), 'predictions to submission.csv')
print()
print(submission.head(8))
print()
print('y_category distribution:')
print(submission['y_category'].value_counts())